# DentalGPT panoramic pipeline (Kaggle runner)

One image goes through three in-distribution steps, all with question shapes taken from the DentalGPT paper:

```text
panoramic X-ray
   -> 14 presence questions (Figure 7 wording, A. True / B. False), whole image (kept as a separate result)
   -> presence_level="region": region by region, the same question for every one of the 14 findings,
      whatever the whole image answered
      (region_prompt="words": "... present in the upper right quadrant of this image", whole image;
       region_prompt="crop": the verbatim question on the region crop) -> present if any region says A; region set
   -> count_level="region": as soon as a region says A for a countable finding, "How many teeth in the upper
      right quadrant ..." there; the finding's count is the sum
      count_level="overall": one whole-image tooth-count question per positive countable finding (Figure 9 wording)
      counting=False: no count question at all; a finding is a presence decision on the whole image and (with
      presence_level="region") in every region, and the evaluation scores presence per finding and per region
   -> question_form="combined" (hosted models): wherever a presence question would be followed by a count question in
      the same scope, one presence-and-count question instead, answered "Answer: A. True / Count: 3" or "Answer:
      B. False / Count: 0"; the five presence-only findings keep the bare question; DentalGPT keeps the separate ones
   -> JSON per image, deterministic dentist summary, TP/FP/TN/FN evaluation, region counts, side check
   -> location truth: ground-truth boxes are translated into the same quadrants by a vision LLM
      (numbered boxes drawn on the image -> FDI quadrant x anterior/posterior), by DentalGPT itself
      (experimental, two multiple-choice questions per box), or by fixed windows
   -> dentist report: a text LLM (the reporter role) gets the findings of one image as one dense JSON (every finding,
      region and count with an explicit status) and returns a classified report in the dentist's language
      (report_language); the reply is verified against the data, corrected once if needed, rendered to Markdown
```

Regions are the two jaws or the four FDI quadrants (patient-side names).

**CELL 3 is the only cell to edit.** It holds one dictionary per configuration: a name plus the knobs that
configuration changes - local or hosted backend, the analyzer / adapter / reporter models, the question shapes,
the region scheme, the location truth. Everything a configuration does not mention comes from
`experiments.DEFAULTS`. Every experiment answers the same images, writes into `<output_root>/<name>/`, and
CELL 10 scores them side by side and ranks them, so one session says which configuration works best instead
of one configuration per session. Run the cells in order; a local first run probes whether the checkpoint
needs the `<think>/<answer>` suffix.

In [ ]:
# ============================================================
# CELL 1 - Python dependencies (llama.cpp is compiled later with CUDA)
# ============================================================
%pip install -q "huggingface_hub>=0.26" "openai>=1.55" "requests>=2.31" "pillow>=10.0" "pandas>=1.5"
print("Python dependencies installed.")

In [ ]:
# ============================================================
# CELL 2 - Locate and import the project files
# ============================================================
import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/dental_x-ray")  # folder holding the project .py files
REQUIRED_PROJECT_FILES = {"dental_pipeline.py", "dental_eval.py", "llama_runtime.py", "location_adapter.py",
                          "llm_api.py", "report_writer.py", "dental_analysis.py", "experiments.py"}
if not PROJECT_DIR.is_dir() and Path.cwd().joinpath("dental_pipeline.py").is_file():
    PROJECT_DIR = Path.cwd()
missing = [f for f in REQUIRED_PROJECT_FILES if not (PROJECT_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing project files in {PROJECT_DIR}: {missing}")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import dental_pipeline as dp
import dental_eval as ev
import dental_analysis as da
import location_adapter as la
import llm_api
import experiments as xp
import report_writer as rw
from llama_runtime import LlamaCppServer, build_llama_cpp, download_dentalgpt
print("PROJECT_DIR =", PROJECT_DIR)
print("Project imports succeeded.")

In [ ]:
# ============================================================
# CELL 3 - CONFIGURATION: the experiments to run and compare (the only cell to edit)
# ============================================================
OUTPUT_ROOT = "/kaggle/working/dental_outputs"

# Provider endpoints and credentials: define each provider once. Keys are read from environment variables or
# Kaggle Secrets (Add-ons > Secrets); missing keys are allowed until that provider is assigned to a role.
PROVIDERS = {
    "openai": {"base_url": "https://api.openai.com/v1",
               "api_key": ""},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",
                   "api_key": ""},
    "nvidia": {"base_url": "https://integrate.api.nvidia.com/v1",
               "api_key": llm_api.secret("NVIDIA_API_KEY", required=False)},
    "gemini": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "api_key": llm_api.secret("GEMINI_API_KEY", required=False)},
}
llm_api.configure_providers(PROVIDERS)

# What every experiment starts from. Any knob of experiments.DEFAULTS may be set here (print(xp.DEFAULTS),
# or read experiments.py, for the full list with its comments): the three model roles, the question shapes,
# the location truth, the report language, the local llama.cpp settings.
SHARED = {
    "output_root": OUTPUT_ROOT,
    # The analyzer answers the finding questions (used when "backend" is "api").
    "analyzer": {"provider": "openrouter", "model": "qwen/qwen3-vl-235b-a22b-thinking",
                 "request_options": {"extra_body": {"provider": {"only": ["novita"],
                                                                 "allow_fallbacks": False}}}},
    # The adapter translates ground-truth boxes into regions (location_truth="llm").
    "adapter": {"provider": "nvidia", "model": "google/gemma-4-31b-it",
                "token_param": "max_completion_tokens", "temperature": None,
                "max_output_tokens": 8192, "max_boxes_per_call": 12},
    # The reporter turns one image's findings into a dentist report (CELL 13). It never sees the image.
    "reporter": {"provider": "nvidia", "model": "google/gemma-4-31b-it",
                 "token_param": "max_completion_tokens", "temperature": None, "max_output_tokens": 8192},
    "report_language": "English",  # every sentence of the report, e.g. "Persian", "German"
}

# One dictionary per configuration: a name, plus only the knobs that configuration changes. Every experiment
# answers the same images and writes into <output_root>/<name>/; CELL 10 ranks them. A dictionary knob merges
# key by key, so {"analyzer": {"model": ...}} keeps the provider and request options set above.
EXPERIMENTS = xp.build([
    {"name": "base"},                                                     # the recommended run
    {"name": "separate-questions", "question_form": "separate"},          # DentalGPT's own question shapes
    {"name": "whole-image-only", "presence_level": "overall", "count_level": "overall"},  # 14 calls per image
    {"name": "presence-only", "counting": False},                         # no counts: presence per finding and region
    # {"name": "arch-regions", "region_scheme": "arch"},                  # two jaws instead of four quadrants
    # {"name": "crop-prompt", "region_prompt": "crop"},                   # the region crop, not the region words
    # {"name": "gemini", "analyzer": {"provider": "gemini", "model": "gemini-3-pro"}},
    # {"name": "no-retry", "parse_retries": 0, "max_tokens": 2048},
    # {"name": "windows-truth", "location_truth": "geometry"},            # no adapter calls
    # {"name": "dentalgpt-local", "backend": "local"},                    # the paper's checkpoint; needs a GPU
    # {"name": "dentalgpt-4k-image", "backend": "local", "image_max_tokens": 4096},
], shared=SHARED)
xp.show(EXPERIMENTS)

# Datasets to run and score; the same images for every experiment, so the comparison is paired.
# kind "yolo": UMFIH 14-class layout; kind "dentex": DENTEX split. "limit": first N images (sorted by id).
DATASETS = [
    {"name": "umfih_test", "kind": "yolo", "limit": 2,
     "images": "/kaggle/working/umfih_14class/data/test/images",
     "labels": "/kaggle/working/umfih_14class/data/test/labels"},
    # {"name": "umfih_external", "kind": "yolo", "limit": None,
    #  "images": "/kaggle/working/umfih_14class/external/images",
    #  "labels": "/kaggle/working/umfih_14class/external/labels"},
    # DENTEX: use the fully labeled train split (705 images, COCO-style JSON) and/or the 50-image
    # validation split (validation_triple.json); DentalGPT never trained on DENTEX, so both are held-out.
    # The 250-image DENTEX test split ships as raw LabelMe files without an official class mapping; skip it.
    # {"name": "dentex_train", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"},
    # {"name": "dentex_val", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/validation_data/quadrant_enumeration_disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/validation_triple.json"},
]

# Machine settings for the local backend: where llama.cpp is built and served, and where the GGUF files are
# cached. The same for every local experiment; what the model sees (checkpoint, context, image tokens) is a
# knob of the experiment instead.
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST, SERVER_PORT, SERVER_ALIAS = "127.0.0.1", 8080, "dentalgpt"
SERVER_LOG_PATH = "/kaggle/working/llama_dentalgpt_server.log"
SERVER_STARTUP_TIMEOUT = 300.0
MODEL_DIR = "/kaggle/working/models/dentalgpt"
HF_TOKEN_SECRET = "HF_TOKEN"  # optional (the GGUF repo is public): environment variable or Kaggle secret
CUDA_ARCH = None              # None = auto-detect (P100 fallback 60)
BUILD_JOBS = 4

LOCAL = [c for c in EXPERIMENTS if xp.is_local(c)]
print(f"\ndatasets = {[d['name'] for d in DATASETS]} | local experiments = {[c['name'] for c in LOCAL]}")

In [ ]:
# ============================================================
# CELL 4 - Environment diagnostics
# ============================================================
import platform
print("Python:", platform.python_version(), "|", platform.platform())


def detect_cuda_arch(fallback="60"):
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                         text=True, stderr=subprocess.STDOUT)
        arch = output.strip().splitlines()[0].strip().replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)
    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = None
if LOCAL:
    for executable in ("git", "cmake", "nvcc", "nvidia-smi"):
        print(f"{executable:12s}:", shutil.which(executable))
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    for executable in ("cmake", "git", "nvcc"):
        if not shutil.which(executable):
            raise RuntimeError(f"{executable} is required to build llama.cpp; enable a GPU accelerator.")
    CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
    print("CUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)
else:
    print("No local experiment; llama.cpp checks skipped.")

In [ ]:
# ============================================================
# CELL 5 - Build or find the pinned llama.cpp server
# ============================================================
LLAMA_SERVER = None
if LOCAL:
    # Reuses an existing build only if it was built from LLAMA_CPP_REF; otherwise rebuilds.
    LLAMA_SERVER = Path(build_llama_cpp(source_dir=LLAMA_CPP_DIR, cuda_arch=CUDA_ARCH_RESOLVED,
                                        jobs=BUILD_JOBS, ref=LLAMA_CPP_REF)).resolve()
    print("llama-server =", LLAMA_SERVER)
else:
    print("No local experiment; llama.cpp build skipped.")

In [ ]:
# ============================================================
# CELL 6 - Local DentalGPT: the GGUF files, and one llama.cpp server at a time
# ============================================================
# Experiments asking for the same checkpoint download it once; experiments asking for the same server
# settings share the running server. A different checkpoint, context or image-token cap restarts it.
import requests

MODELS = {}
for cfg in LOCAL:
    key = xp.model_key(cfg)
    if key in MODELS:
        continue
    repo_id, model_filename, mmproj_filename = key
    MODELS[key] = download_dentalgpt(model_dir=MODEL_DIR, repo_id=repo_id, model_filename=model_filename,
                                     mmproj_filename=mmproj_filename,
                                     hf_token=llm_api.secret(HF_TOKEN_SECRET, required=False))
    for label, path in (("Language model", MODELS[key].model_path), ("Vision projector", MODELS[key].mmproj_path)):
        print(f"{label}: {path} ({Path(path).stat().st_size / 1024**3:.2f} GiB)")

SERVER = SERVER_SETTINGS = None


def local_server(cfg):
    """The llama.cpp server for one experiment, restarted only when its local settings change."""
    global SERVER, SERVER_SETTINGS
    if SERVER is not None and SERVER_SETTINGS == xp.server_key(cfg):
        return SERVER
    if SERVER is not None:
        SERVER.stop()
    files = MODELS[xp.model_key(cfg)]
    SERVER = LlamaCppServer(binary=LLAMA_SERVER, model_path=files.model_path, mmproj_path=files.mmproj_path,
                            host=SERVER_HOST, port=SERVER_PORT, alias=SERVER_ALIAS,
                            n_gpu_layers=cfg["n_gpu_layers"], ctx_size=cfg["ctx_size"],
                            image_max_tokens=cfg["image_max_tokens"], image_min_tokens=cfg["image_min_tokens"],
                            startup_timeout=SERVER_STARTUP_TIMEOUT, log_path=SERVER_LOG_PATH)
    SERVER.start(reuse_existing=False)
    ids = [m.get("id") for m in requests.get(f"{SERVER.base_url}/v1/models", timeout=10).json().get("data", [])]
    if SERVER_ALIAS not in ids:
        raise RuntimeError(f"Expected alias {SERVER_ALIAS!r}; /v1/models returned {ids}. See {SERVER_LOG_PATH}.")
    SERVER_SETTINGS = xp.server_key(cfg)
    print(f"{cfg['name']}: DentalGPT server verified at {SERVER.base_url}")
    return SERVER


def open_runner(cfg):
    """The analyzer of one experiment: its hosted model, or the local server started above."""
    return xp.runner(cfg, local_server(cfg) if xp.is_local(cfg) else None)


print("Model files ready:", len(MODELS), "| local server helper defined")

In [ ]:
# ============================================================
# CELL 7 - Load ground truth for every dataset (no model calls)
# ============================================================
GT = {}
for spec in DATASETS:
    if spec["kind"] == "yolo":
        gt = ev.load_yolo(spec["images"], spec["labels"])
    elif spec["kind"] == "dentex":
        gt = ev.load_dentex(spec["images"], spec["annotations"])
    else:
        raise ValueError(f"unknown dataset kind {spec['kind']!r}")
    if spec.get("limit"):
        gt = dict(sorted(gt.items())[: spec["limit"]])
    GT[spec["name"]] = gt
    positives = sum(1 for g in gt.values() if g["boxes"])
    print(f"{spec['name']}: {len(gt)} images, {positives} with at least one finding, "
          f"{sum(len(g['boxes']) for g in gt.values())} boxes")

In [ ]:
# ============================================================
# CELL 8 - Run every experiment (resumable: finished images are skipped)
# ============================================================
# One experiment at a time, one directory each: <output_root>/<name>/<dataset>/. The resolved configuration
# is saved as experiment.json and hashed into the run manifest, so a changed knob can never be mixed into a
# resumed run. An experiment that fails is reported and the sweep continues with the next one.
import traceback

MODE_USED, FAILED = {}, {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    print(f"\n{'=' * 78}\n=== {name}: {xp.analyzer_name(cfg)} | {xp.protocol(cfg)}\n{'=' * 78}")
    try:
        xp.record(cfg)
        runner = open_runner(cfg)
        paths = [g["path"] for _, g in sorted(next(iter(GT.values())).items())]
        MODE_USED[name], probe = xp.resolve_mode(cfg, runner, paths, [d["name"] for d in DATASETS])
        if probe:
            for probe_mode in dp.MODES:
                s = probe[probe_mode]
                print(f"  probe {probe_mode:7s}: think tags {s['think_tag_rate']:.0%} | answer tags "
                      f"{s['answer_tag_rate']:.0%} | parsed {s['parse_rate']:.0%} | truncated {s['truncation_rate']:.0%}")
        print(f"  mode = {MODE_USED[name]} | runner = {runner.settings()}")
        for spec in DATASETS:
            images = {image_id: g["path"] for image_id, g in GT[spec["name"]].items()}
            out = dp.run_dataset(runner, images, xp.run_dir(cfg, spec["name"]), mode=MODE_USED[name],
                                 protocol=xp.protocol(cfg), resume=True, provenance=xp.provenance(cfg, llama_cpp_ref=LLAMA_CPP_REF))
            print("  saved:", out)
    except Exception:
        FAILED[name] = traceback.format_exc()
        print(f"!! {name} FAILED; the other experiments continue\n{FAILED[name]}")

print("\nfinished:", [c["name"] for c in EXPERIMENTS if c["name"] not in FAILED], "| failed:", sorted(FAILED))

In [ ]:
# ============================================================
# CELL 9 - Location truth: translate ground-truth boxes into the region windows (resumable)
# ============================================================
# Independent of the model run, and keyed by the adapter rather than by the experiment: every experiment
# using the same adapter reads the same translated boxes from
# <output_root>/location_truth/<dataset>/<adapter>/ instead of paying for them again.
# One JSON per image under boxes/, drawn images under drawn/ for audit; unparseable replies follow
# location_failure_policy, and every attempt and fallback is saved.
ADAPTED, TRUTH_DIRS = {}, {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    for spec in DATASETS:
        dataset = spec["name"]
        if not cfg["evaluate_location"] or cfg["location_truth"] == "geometry" or not xp.protocol(cfg).uses_regions:
            print(f"{name}/{dataset}: fixed windows (and FDI quadrant labels where the dataset has them)")
            continue
        out = xp.truth_dir(cfg, dataset, MODE_USED.get(name))
        if out in TRUTH_DIRS:
            ADAPTED[name, dataset] = TRUTH_DIRS[out]
            print(f"{name}/{dataset}: reuses {out}")
            continue
        adapter = xp.location_adapter(cfg, open_runner(cfg) if cfg["location_truth"] == "fdm" else None,
                                      mode=MODE_USED.get(name, "plain"))
        TRUTH_DIRS[out] = ADAPTED[name, dataset] = la.adapt_dataset(adapter, GT[dataset], out, resume=True)
        print(f"{name}/{dataset}: {la.summarize(ADAPTED[name, dataset])}")
        agreement = ev.truth_agreement(GT[dataset], ADAPTED[name, dataset])
        if agreement["boxes_with_fdi"]:
            # DENTEX carries FDI quadrant labels: exact windows, so this is the adapter's own accuracy.
            print(f"{name}/{dataset}: adapter vs FDI truth {agreement}")

In [ ]:
# ============================================================
# CELL 10 - Evaluate every experiment and rank them
# ============================================================
import pandas as pd
from IPython.display import display

# Each experiment is scored against its own location truth and written to <name>/<dataset>/evaluation/.
REPORTS = {}
for cfg in EXPERIMENTS:
    for spec in DATASETS:
        name, dataset = cfg["name"], spec["name"]
        results = dp.load_results(xp.run_dir(cfg, dataset))
        if not (set(GT[dataset]) & set(results)):
            print(f"{name}/{dataset}: no results under {xp.run_dir(cfg, dataset)}; run CELL 8 first.")
            continue
        truth = ev.apply_adapted(GT[dataset], ADAPTED[name, dataset]) if (name, dataset) in ADAPTED else GT[dataset]
        REPORTS[name, dataset] = ev.evaluate(truth, results, dataset=dataset,
                                             out_dir=xp.run_dir(cfg, dataset) / "evaluation",
                                             evaluate_location=cfg["evaluate_location"])

# The leaderboard: one row per experiment and dataset, best F1 first.
rows = []
for (name, dataset), report in REPORTS.items():
    summary, extra = report["summary"], da.metrics(report)
    rows.append({"dataset": dataset, "experiment": name, "images": summary["images_scored"],
                 **{k: summary[k] for k in ("f1", "sensitivity", "specificity", "ppv", "macro_f1",
                                            "mean_false_alarms_per_image", "unparseable_rate")},
                 "coverage": extra["coverage"], "counts_mae": extra["counts_mae"],
                 "regions_exact": extra["regions_exact_set_match_rate"], "region_f1": extra["region_presence_f1"],
                 "calls_per_image": summary["mean_calls_per_image"]})
LEADERBOARD = pd.DataFrame(rows).sort_values(["dataset", "f1"], ascending=[True, False])
if not LEADERBOARD.empty:
    Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
    LEADERBOARD.to_csv(Path(OUTPUT_ROOT) / "leaderboard.csv", index=False)
    display(LEADERBOARD.set_index(["dataset", "experiment"]))

# Paired comparison against the first complete experiment: presence and counts on exactly the same images,
# so the columns say what a knob changed (corrected/worsened checks), not what the image sample was.
# Location is left out here because each experiment has its own location truth in the table above.
COMPARISONS = {}
for spec in DATASETS:
    dataset = spec["name"]
    complete = {c["name"]: xp.run_dir(c, dataset) for c in EXPERIMENTS
                if (c["name"], dataset) in REPORTS and not REPORTS[c["name"], dataset]["missing_results"]}
    if len(complete) < 2:
        continue
    COMPARISONS[dataset] = da.compare_runs(GT[dataset], complete, dataset=dataset, evaluate_location=False)
    ev.write_report(COMPARISONS[dataset], Path(OUTPUT_ROOT) / "comparison" / dataset)
    print(f"\n===== {dataset}: paired against {next(iter(complete))} =====")
    display(pd.DataFrame(COMPARISONS[dataset]["run_comparison"])[
        ["run", "model", "presence_level", "counting", "count_level", "region_scheme", "region_prompt", "question_form",
         "paired_checks", "paired_reference_f1", "paired_run_f1", "paired_f1_delta", "corrected", "worsened",
         "newly_unresolved", "mean_calls_per_image", "completion_tokens"]].set_index("run"))

In [ ]:
# ============================================================
# CELL 11 - One experiment in detail: TP/FP/TN/FN per finding, counts, regions, side check
# ============================================================
INSPECT_EXPERIMENT = EXPERIMENTS[0]["name"]  # any name from CELL 3
INSPECT_DATASET = DATASETS[0]["name"]

report = REPORTS.get((INSPECT_EXPERIMENT, INSPECT_DATASET))
if report is None:
    print(f"No evaluation for {INSPECT_EXPERIMENT}/{INSPECT_DATASET}; run CELLs 8 and 10 first.")
else:
    print(f"===== {INSPECT_EXPERIMENT} / {INSPECT_DATASET} =====")
    display(pd.DataFrame([report["summary"]]).T)
    display(pd.DataFrame(report["presence"]).set_index("condition"))
    if report["whole_image"]:
        print("whole-image answers alone (what the regional pass recovered, and what it cost):")
        display(pd.DataFrame(report["whole_image"]).set_index("condition"))
    if report["counts"]:
        display(pd.DataFrame(report["counts"]).set_index("condition"))
    if report["region_counts"]:
        display(pd.DataFrame(report["region_counts"]).set_index(["condition", "region"]))
    if report["region_presence"]:
        print("presence per region (every region answer of every image, whatever the whole image said):")
        display(pd.DataFrame(report["region_presence"]).set_index(["condition", "region"]))
    if report["regions"]:
        display(pd.DataFrame(report["regions"]).set_index("condition"))
    for table, index in (("case_breakdown", ["situation", "group"]), ("stage_changes", ["condition", "transition"]),
                         ("parse_recovery", ["stage", "outcome"]), ("call_usage", ["stage", "attempt"])):
        rows = report.get(table)
        if rows:
            frame = pd.DataFrame(rows)
            keys = [k for k in index if k in frame.columns]
            print(f"\n{table}:")
            display(frame.set_index(keys) if keys else frame)

In [ ]:
# ============================================================
# CELL 12 - Inspect one image: dentist summary and raw model answers
# ============================================================
INSPECT_IMAGE = None   # None = first image of the dataset
SHOW_RAW_CALLS = False

cfg = next(c for c in EXPERIMENTS if c["name"] == INSPECT_EXPERIMENT)
results = dp.load_results(xp.run_dir(cfg, INSPECT_DATASET))
image_id = INSPECT_IMAGE or next(iter(sorted(results)))
result = results[image_id]
print(f"{INSPECT_EXPERIMENT} / {INSPECT_DATASET} / {image_id}\n")
print(dp.dentist_report(result))
truth = [b["condition"] for b in GT[INSPECT_DATASET][image_id]["boxes"]]
print("\nGround-truth boxes:", {c: truth.count(c) for c in dict.fromkeys(truth)} or "none")
adapted = ADAPTED.get((INSPECT_EXPERIMENT, INSPECT_DATASET))
if adapted:
    for k, record in enumerate(adapted[image_id]["boxes"], 1):
        print(f"  box {k}: {record['condition']} -> {record['regions']} ({record['source']}; fixed windows {record['geometry']})")
if SHOW_RAW_CALLS:
    for call in result["calls"]:
        print(f"\n--- {call['stage']} | {call['condition']} | {call['region']} | finish={call['finish_reason']}")
        print(call["text"][:800])

In [ ]:
# ============================================================
# CELL 13 - Dentist report: one LLM call per image over the structured findings (resumable)
# ============================================================
# The analyzer answered up to 14 + R x 14 (+ counts) narrow questions per image. The report writer gets them as
# one dense JSON (every finding, every region, every count, each with an explicit status), returns a report in
# the experiment's report_language as JSON (one entry per finding in seven sections, impression, caveats), which
# is verified against the data (every finding exactly once, statuses unchanged, nothing invented), sent back once
# for correction if it fails, and rendered to Markdown. One .json + one .md per image under
# <name>/<dataset>/reports/<model>-<language>/reports; a reply that fails twice keeps the deterministic dentist
# summary, marked as such. The model never sees the image.
from IPython.display import Markdown, display

REPORT_EXPERIMENTS = [INSPECT_EXPERIMENT]  # one call per image per experiment; [c["name"] for c in EXPERIMENTS] for all

WRITTEN = {}
for cfg in [c for c in EXPERIMENTS if c["name"] in REPORT_EXPERIMENTS]:
    writer = xp.report_writer(cfg)
    print(f"{cfg['name']}: report writer {writer.public()}")
    for spec in DATASETS:
        name, dataset = cfg["name"], spec["name"]
        results = dp.load_results(xp.run_dir(cfg, dataset))
        if not results:
            print(f"{name}/{dataset}: no results; run CELL 8 first.")
            continue
        WRITTEN[name, dataset] = rw.report_dataset(
            writer, results, xp.run_dir(cfg, dataset) / "reports" / writer.run_name,
            analyzer=xp.analyzer_name(cfg), resume=True, limit=cfg["report_images"])
        print(f"{name}/{dataset}: {rw.summarize_reports(WRITTEN[name, dataset])}")

# One report to read (the image inspected in CELL 12 when it has one).
reports = WRITTEN.get((INSPECT_EXPERIMENT, INSPECT_DATASET)) or next(iter(WRITTEN.values()), {})
if reports:
    shown = reports.get(globals().get("image_id")) or reports[next(iter(sorted(reports)))]
    print(f"{shown['image_id']}: verified={shown['verified']} | attempts={len(shown['attempts'])} | "
          f"problems={shown['problems']}")
    display(Markdown(shown["markdown"]))